# D2 — Muat 5 tabel gold D1 ke BigQuery milikmu

Di D1 kita berhenti di **gold layer** di data lake. Di D2 kita mulai dari sana: lima tabel gold yang sama kita muat ke **warehouse** (BigQuery) supaya bisa di-query dengan SQL dan divisualisasikan di Data Studio.

**Yang kamu lakukan hanya 3 hal:**
1. Isi `PROJECT_ID` di sel pertama (satu baris saja).
2. Jalankan sel-sel berikutnya berurutan (Shift + Enter). Saat diminta, izinkan akses akun Google-mu.
3. Pastikan sel terakhir menampilkan **semua tabel ✅**.

Tabel akan ada di `PROJECT_ID.d1_gold`. Menjalankan ulang aman: tabel ditimpa, bukan digandakan.

In [ ]:
# === 1. Konfigurasi — satu-satunya sel yang perlu kamu ubah ===
PROJECT_ID = "ISI-PROJECT-ID-DI-SINI"   # contoh: "trial-trialan-lah" (lihat di pojok kiri atas Google Cloud Console)

# Dua baris di bawah ini disiapkan instruktur, jangan diubah.
BASE_URL = "https://raw.githubusercontent.com/riztekur/gobel-d2-data/main/d2-data/parquet"
DATASET  = "d1_gold"

# Nama tabel dan jumlah baris yang HARUS kamu dapatkan (angka ini sama dengan output D1).
EXPECTED_ROWS = {
    "dim_product": 120,
    "dim_dealer": 180,
    "fct_sales_daily": 239_706,
    "mart_sales_region_monthly": 10_304,
    "mart_inventory_coverage": 117,
}

assert PROJECT_ID != "ISI-PROJECT-ID-DI-SINI", "Isi PROJECT_ID dulu, lalu jalankan sel ini lagi."
assert "GANTI" not in BASE_URL, "BASE_URL belum diisi instruktur."
print("Konfigurasi siap ->", PROJECT_ID)

In [ ]:
# === 2. Masuk dengan akun Google-mu ===
# Akan muncul jendela izin. Pilih akun yang SAMA dengan yang kamu pakai di BigQuery.
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)

# Dataset = "folder" berisi tabel-tabelmu. exists_ok=True: kalau sudah ada, tidak error.
client.create_dataset(bigquery.Dataset(f"{PROJECT_ID}.{DATASET}"), exists_ok=True)
print(f"Dataset {PROJECT_ID}.{DATASET} siap")

In [ ]:
# === 3. Muat 5 tabel ===
# Tiap tabel: unduh 1 file Parquet dari repo -> kirim ke BigQuery sebagai load job.
# Parquet menyimpan tipe data (angka, tanggal) apa adanya, jadi tidak perlu mengatur skema.
import io, urllib.request

for nama in EXPECTED_ROWS:
    isi = urllib.request.urlopen(f"{BASE_URL}/{nama}.parquet").read()      # unduh file
    job = client.load_table_from_file(
        io.BytesIO(isi),
        f"{PROJECT_ID}.{DATASET}.{nama}",
        job_config=bigquery.LoadJobConfig(
            source_format=bigquery.SourceFormat.PARQUET,
            write_disposition="WRITE_TRUNCATE",   # timpa kalau tabel sudah ada -> aman dijalankan ulang
        ),
    )
    job.result()                                  # tunggu sampai selesai
    print(f"  dimuat: {nama:<28} {len(isi)/1e6:6.2f} MB")

In [ ]:
# === 4. Periksa hasilnya ===
# Bandingkan jumlah baris di BigQuery dengan angka yang seharusnya.
semua_ok = True
for nama, seharusnya in EXPECTED_ROWS.items():
    aktual = client.get_table(f"{PROJECT_ID}.{DATASET}.{nama}").num_rows
    ok = aktual == seharusnya
    semua_ok &= ok
    print(("✅" if ok else "❌"), f"{nama:<28} {aktual:>9,} baris", "" if ok else f"(seharusnya {seharusnya:,})")

print()
if semua_ok:
    print("Semua tabel siap. Buka BigQuery:")
    print(f"https://console.cloud.google.com/bigquery?project={PROJECT_ID}")
else:
    print("Ada yang tidak cocok. Jalankan sel 3 sekali lagi; kalau masih ❌, angkat tangan.")